In [1]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("/content/gdrive/MyDrive/Colab Notebooks/Fake_News_Detection_Cleaned.csv")

# PreProcessing

In [4]:
#Converting to Lowercase
df["text"]=df["text"].str.lower()
df["title"]=df["title"].str.lower()

In [5]:
import re

In [8]:
#Fill Missing values
df["text"] = df["text"].fillna("")
df["title"] = df["title"].fillna("")

In [9]:
#Removing Punctuation
def remove_punctuation(text):
  text=re.sub(r"[^A-Za-z0-9\s]","",text)
  return text

df["text"]=df["text"].apply(remove_punctuation)
df["title"]=df["title"].apply(remove_punctuation)

In [10]:
#Remove all extra spaces: leading, Trailing and in middle
df["text"]=df["text"].str.replace(r"\s+"," ",regex=True).str.strip()
df["title"]=df["title"].str.replace(r"\s+"," ",regex=True).str.strip()

In [11]:
#Remove dataset specific Noise
noise_words=[
    "watch","video","factbox","breaking","update","bbc","live","report","reports"
]

def remove_noise(text):
  pattern=r'\b('+'|'.join(noise_words)+r')\b'
  text=re.sub(pattern,"",text)
  return text

df["text"]=df["text"].apply(remove_noise)
df["title"]=df["title"].apply(remove_noise)

In [18]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [19]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [20]:
#Remove Stop Words
def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
  return text

df["text"]=df["text"].apply(remove_stopwords)
df["title"]=df["title"].apply(remove_stopwords)

In [21]:
df.head()

,Unnamed: 0,title,text,label
0,0,law enforcement high alert following threats ...,comn expeced brck obm mr fyf911 fukylg ...,1
1,1,,post votes hillary already,1
2,2,unbelievable obamas attorney general says cha...,demonstrrs gred lst night exercisg ir con...,1
3,3,bobby jindal raised hindu uses sry christian ...,dozen poltclly ctve psrs c prvte dner frdy...,0
4,4,sat 2 russia unvelis image terrifying new s...,r28 rm miile dubbed n 2 replce 18 flie 43 ...,1


In [22]:
#Stemming
from nltk.stem import PorterStemmer

In [25]:
def stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]

  tokens=word_tokenize(text)
  for token in tokens:
    stemmed_token=ps.stem(token)
    stemmed_words.append(stemmed_token)
  return " ".join(stemmed_words)

df["text"]=df["text"].apply(stemming)
df["title"]=df["title"].apply(stemming)

In [26]:
df.head()

,Unnamed: 0,title,text,label
0,0,law enforc high alert follow threat cop white ...,comn expec brck obm mr fyf911 fukylg blcklver ...,1
1,1,,post vote hillari alreadi,1
2,2,unbeliev obama attorney gener say charlott rio...,demonstrr gred lst night exercisg ir constitut...,1
3,3,bobbi jindal rais hindu use sri christian conv...,dozen poltclli ctve psr c prvte dner frdi nght...,0
4,4,sat 2 russia unv imag terrifi new supernuk wes...,r28 rm miil dub n 2 replc 18 flie 43 mile 7km ...,1


In [27]:
#Combine text and title

df["content"]=df["title"].fillna('')+" "+df["text"].fillna(' ')

In [29]:
#Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df['content'])

In [30]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10471952 stored elements and shape (72095, 5000)>

# Data Set and Data Loaders

In [32]:
y=df['label']

In [33]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [34]:
X_train.shape

(57676, 5000)

In [35]:
X_test.shape

(14419, 5000)

In [36]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [37]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [39]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [41]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

# BUILD OUR RNN

In [42]:
import torch.nn as nn
import torch.optim as optim

In [43]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super().__init__()

    self.hidden_size=hidden_size
    self.num_layers=num_layers

    self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    self.fc=nn.Linear(hidden_size,1)

  def forward(self,x):
    h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

    out,_=self.rnn(x,h0)

    out=self.fc(out[:,-1,:])
    return out

In [44]:
input_size=X_train.shape[1]

model=RNN(input_size)

criterion=torch.nn.BCELoss()
optimizer=optim.Adam(model.parameters())

# Training the RNN

In [46]:
epochs=10
for epoch in range(epochs):
  model.train()

  for Xb,yb in train_loader:
    optimizer.zero_grad()

    Xb=Xb.unsqueeze(1)
    outputs=model(Xb)

    outputs=torch.sigmoid(outputs.squeeze())

    loss=criterion(outputs,yb)
    loss.backward()
    optimizer.step()

  print(f"epoch={epoch+1}/{epochs} and loss={loss.item()}")

epoch=1/10 and loss=0.10768362879753113
epoch=2/10 and loss=0.017200129106640816
epoch=3/10 and loss=0.19137239456176758
epoch=4/10 and loss=0.0052651334553956985
epoch=5/10 and loss=0.004666553344577551
epoch=6/10 and loss=0.012340448796749115
epoch=7/10 and loss=0.012726991437375546
epoch=8/10 and loss=0.02769213356077671
epoch=9/10 and loss=0.03158898279070854
epoch=10/10 and loss=0.0063706920482218266


In [47]:
model.eval()
with torch.no_grad():
  correct_vals=0
  total_vals=0

  for Xb,yb in test_loader:
    Xb=Xb.unsqueeze(1)
    outputs=model(Xb)
    predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()

    total_vals+=yb.size(0)
    correct_vals+=(predicted==yb).sum().item()

  print(f"accuracy={correct_vals/total_vals *100}")

accuracy=95.23545322144393


Why threshold = 0.5?

Because sigmoid gives:

0 → fake
1 → real

So:

≥ 0.5 → class 1
< 0.5 → class 0

In [89]:
#Converting probabilities to prediction for evaluation of metrics
y_pred=(outputs>=0.5).int()

In [90]:
from sklearn.metrics import precision_score,recall_score,f1_score

print(precision_score(yb,y_pred)*100)
print(recall_score(yb,y_pred)*100)
print(f1_score(yb,y_pred)*100)

92.85714285714286
100.0
96.29629629629629


# Model Comparison

In [53]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

In [68]:
X.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [69]:
X=tf.fit_transform(df['content'])
y=df["label"]
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [70]:
model=LogisticRegression()
model.fit(X_train,y_train)

LogisticRegression()

In [71]:
y_pred=model.predict(X_test)

In [72]:
from sklearn.metrics import accuracy_score

In [73]:
print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(f1_score(y_test,y_pred))

0.9477078854289479
0.9469442578912022
0.951545417735187
0.949239262151609


In [77]:
#Naive Bayes
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB

In [78]:
gnb_model=GaussianNB()
gnb_model = MultinomialNB()
gnb_model.fit(X_train,y_train)

MultinomialNB()

In [79]:
y_pred=model.predict(X_test)

In [80]:
print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(f1_score(y_test,y_pred))

0.9477078854289479
0.9469442578912022
0.951545417735187
0.949239262151609


In [84]:
results = {
    "Model": ["Logistic Regression", "Naive Bayes", "RNN"],
    "Accuracy": [0.94, 0.94, 0.95],
    "Precision": [0.94, 0.94, 0.92],
    "Recall": [0.95, 0.95, 1],
    "F1-score": [0.94, 0.94, 0.96]
}
df_results=pd.DataFrame(results)
df_results.sort_values(by="F1-score", ascending=False)

,Model,Accuracy,Precision,Recall,F1-score
2,RNN,0.95,0.92,1.00,0.96
0,Logistic Regression,0.94,0.94,0.95,0.94
1,Naive Bayes,0.94,0.94,0.95,0.94
